# AI Specialist ASR - Colab Runner

## Goal
GitHub의 동일 코드베이스를 사용해 공개 한국어 음성 샘플을 자동 준비하고 Baseline과 Whisper LoRA 학습을 실행합니다. 데이터, checkpoint, 실행 이력, Word 보고서는 Google Drive에 보존합니다.

## Setup

런타임 유형을 **T4 GPU** 이상으로 바꾼 뒤 위에서부터 순서대로 실행합니다. 공개 Zeroth-Korean 샘플만 사용하므로 별도 음성 파일은 필요하지 않습니다. 실제 사내 민감 데이터는 승인 없이 Colab에 업로드하지 마세요.

In [ ]:
GITHUB_REPO_URL = "https://github.com/Pronesis9758/aias-specialist-asr.git"
GITHUB_BRANCH = "codex/initial-asr-automation"  # PR 병합 후 main으로 변경
PROJECT_DIR = "/content/AIAS"
DRIVE_ROOT = "/content/drive/MyDrive/AI_Specialist_ASR_Project"
CONFIG = "configs/colab_public_sample.yaml"

## Steps

### 1. GPU와 Drive 연결

In [ ]:
!nvidia-smi
from google.colab import drive

drive.mount("/content/drive")

### 2. GitHub 코드 동기화

공개 저장소는 그대로 실행됩니다. 비공개 저장소라면 Colab 왼쪽 열쇠 아이콘의 Secrets에 `GITHUB_TOKEN`을 추가하고 이 노트북의 액세스를 켜세요. 토큰에는 이 저장소를 읽을 수 있는 최소 권한만 부여합니다.

In [ ]:
import os, stat, subprocess
from pathlib import Path

clone_env = os.environ.copy()
askpass = Path("/content/git-askpass.sh")
try:
    from google.colab import userdata

    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = None
if token:
    askpass.write_text(
        "#!/bin/sh\n"
        'case "$1" in\n'
        '  *Username*) echo "x-access-token";;\n'
        '  *) echo "$GITHUB_TOKEN";;\n'
        "esac\n"
    )
    askpass.chmod(stat.S_IRUSR | stat.S_IWUSR | stat.S_IXUSR)
    clone_env.update(
        {
            "GIT_ASKPASS": str(askpass),
            "GIT_TERMINAL_PROMPT": "0",
            "GITHUB_TOKEN": token,
        }
    )
try:
    if not os.path.exists(PROJECT_DIR):
        subprocess.run(
            [
                "git",
                "clone",
                "--branch",
                GITHUB_BRANCH,
                "--single-branch",
                GITHUB_REPO_URL,
                PROJECT_DIR,
            ],
            env=clone_env,
            check=True,
        )
    else:
        subprocess.run(
            ["git", "-C", PROJECT_DIR, "fetch", "origin", GITHUB_BRANCH],
            env=clone_env,
            check=True,
        )
        subprocess.run(
            ["git", "-C", PROJECT_DIR, "checkout", GITHUB_BRANCH],
            env=clone_env,
            check=True,
        )
        subprocess.run(
            [
                "git",
                "-C",
                PROJECT_DIR,
                "merge",
                "--ff-only",
                f"origin/{GITHUB_BRANCH}",
            ],
            env=clone_env,
            check=True,
        )
finally:
    if askpass.exists():
        askpass.unlink()
os.chdir(PROJECT_DIR)
result = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True, check=True)
print("Git commit:", result.stdout.strip())

### 3. 의존성 설치

In [ ]:
# Colab 이미지를 기준으로 설치합니다. 기본 CUDA PyTorch는 유지하고,
# Colab에 따라 함께 설치되는 torchao는 Whisper/PEFT와 충돌할 수 있어 제거합니다.
%pip uninstall -y torchao
%pip install -q -e ".[train]" "transformers>=4.46,<5" "peft>=0.14,<0.19"
import sys, torch, transformers, peft

print(
    {
        "python": sys.executable,
        "torch": torch.__version__,
        "cuda": torch.cuda.is_available(),
        "transformers": transformers.__version__,
        "peft": peft.__version__,
    }
)

### 4. 공개 한국어 음성 샘플 준비

CC BY 4.0 Zeroth-Korean의 고정된 리비전에서 학습 40·검증 8·테스트 16개를 스트리밍합니다. 두 번째 실행부터는 Drive의 완성된 manifest와 오디오를 재사용합니다.

In [ ]:
!{sys.executable} -m aias_specialist.cli prepare-hf-dataset --config "{CONFIG}"

### 5. 모델 버전 고정과 Baseline 실행

In [ ]:
!{sys.executable} -m aias_specialist.cli doctor --config "{CONFIG}"
!{sys.executable} -m aias_specialist.cli model-lock --config "{CONFIG}"
!{sys.executable} -m aias_specialist.cli download-model --config "{CONFIG}"
!{sys.executable} -m aias_specialist.cli run --config "{CONFIG}"

### 6. Whisper LoRA 학습

In [ ]:
!{sys.executable} -m aias_specialist.cli train-whisper --config "{CONFIG}"

## Checks

Drive에 결과와 checkpoint가 남았는지 확인합니다.

In [ ]:
import json
from pathlib import Path

for required in [
    Path(DRIVE_ROOT) / "artifacts/runs",
    Path(DRIVE_ROOT) / "backdata/experiments.sqlite3",
    Path(DRIVE_ROOT) / "checkpoints",
]:
    print(required, "OK" if required.exists() else "MISSING")

training_runs = sorted((Path(DRIVE_ROOT) / "artifacts/runs").glob("train-*"))
if training_runs:
    latest = training_runs[-1]
    result = json.loads((latest / "metrics.json").read_text(encoding="utf-8"))
    print("Latest training run:", latest.name)
    print("Base test WER:", result["baseline"]["wer"])
    print("LoRA test WER:", result["lora"]["wer"])
    reduction = result["lora_improvement"]["wer_absolute_reduction"]
    print("WER absolute reduction:", reduction)
    print("Report:", latest / "reports/evaluation_report.docx")

## Next Steps

이 노트북은 데이터 준비, Baseline 실행, LoRA 학습, 동일 test split의 Base Whisper·Best LoRA 비교, SQLite 등록, Word 보고서 생성까지 자동화한 소규모 기술 검증입니다. 공개 데이터가 일반 한국어이므로 제조 현장 성능 근거로 사용하지 않습니다. 실제 현장 음성·정답 문장을 승인된 저장소에 준비한 후 동일한 manifest 스키마로 교체하고, 최종 모델 선택과 보고서 결론은 현업 담당자가 검토합니다.